# 02 - Basic Preprocessing

Basic preprocessing of the CIC-DDoS2019 dataset for binary classification.

This notebook cleans the data, handles missing values, encodes the target variable, and saves a processed sample ready for machine learning.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
# Configuration
DATA_DIR = "../data/raw/CSVs"
SAMPLE_ROWS = 5000  # rows per CSV file
RANDOM_STATE = 42

# Find all CSV files
csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)

print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f"  - {f}")

## Load Data (with sampling)


In [ ]:
# Check if CSV files were found
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

In [ ]:
def load_and_sample(filepath, n_rows=SAMPLE_ROWS, random_state=RANDOM_STATE):
    """Load CSV with optional sampling."""
    df = pd.read_csv(filepath, low_memory=False)
    
    if len(df) > n_rows:
        return df.sample(n=n_rows, random_state=random_state)
    
    return df

In [ ]:
# Load all files
dfs = []

for f in csv_files:
    print(f"Loading {os.path.basename(f)}...")
    df_part = load_and_sample(f)
    dfs.append(df_part)
    print(f"  -> {len(df_part)} rows")

# Combine
df = pd.concat(dfs, ignore_index=True)

print(f"\nTotal: {len(df):,} rows, {df.shape[1]} columns")
df.head()

## Basic Data Overiew


In [ ]:
# Basic info
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nShape:")
print(df.shape)

## Clean Columns Names


In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

In [ ]:
# Check target column
if "Label" not in df.columns:
    raise KeyError("Column 'Label' was not found in the dataset.")

## Remove Duplicates


In [ ]:
# Remove duplicate rows
rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Rows after removing duplicates: {rows_after:,}")
print(f"Removed duplicates: {rows_before - rows_after:,}")

## Handle Infinite Values


In [ ]:
# Detect numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

print(f"Numeric columns: {len(numeric_cols)}")

In [ ]:
# Count infinite values
inf_count = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values before cleaning: {inf_count}")

In [ ]:
# Replace infinite values with NaN
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

inf_count_after = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values after cleaning: {inf_count_after}")

## Handle Missing Values


In [ ]:
# Missing values before cleaning
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing_values)

In [ ]:
# Drop rows with missing values
rows_before = len(df)

df = df.dropna()

rows_after = len(df)

print(f"Rows before dropping missing values: {rows_before:,}")
print(f"Rows after dropping missing values: {rows_after:,}")
print(f"Removed rows: {rows_before - rows_after:,}")

In [ ]:
# Verify missing values
print(f"Total missing values after cleaning: {df.isna().sum().sum()}")

## Encode Target Variable


In [ ]:
# Original label distribution
print("Original labels:")
print(df["Label"].value_counts())

In [ ]:
# Encode target for binary classification
# BENIGN -> 0
# DDoS attacks -> 1
df["target"] = df["Label"].apply(lambda label: 0 if label == "BENIGN" else 1)

print("Binary target distribution:")
print(df["target"].value_counts())

print("\nBinary target distribution (%):")
print(df["target"].value_counts(normalize=True) * 100)

In [ ]:
# Drop original Label column
df = df.drop(columns=["Label"])

print(f"Columns after target encoding: {df.shape[1]}")

## Encode Categorical Features


In [ ]:
# Detect categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}")
for col in categorical_cols:
    print(f"  - {col}")

In [ ]:
# One-hot encode categorical columns
if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print("Categorical columns encoded.")
else:
    print("No categorical columns to encode.")

print(f"Shape after encoding: {df.shape}")

In [ ]:
# Convert boolean columns to integers
bool_cols = df.select_dtypes(include=["bool"]).columns

df[bool_cols] = df[bool_cols].astype(int)

print(f"Converted {len(bool_cols)} boolean columns to integers.")

## Final Check


In [ ]:
# Check infinite values after preprocessing
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values after preprocessing: {inf_count}")

In [ ]:
# Final dataset check
print("Final shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes.value_counts())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nTarget distribution:")
print(df["target"].value_counts())

In [ ]:
df.head()

## Save Processed Data


In [ ]:
# Save processed data
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/ddos_basic_preprocessed.csv"

df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows to {output_path}")